# TransportTech — velosiped şəbəkəsinin geofəza təhlili

Bu notebook **“Hackathon Seçim Tapşırığı V2”** sənədinin üç hesablama sualını
və AYNA rəhbərliyinə strateji təklifini cavablandırır. Nəticələr bu qovluqda
verilən KMZ fayllarına əsaslanır. İnternetdən yeni məlumat endirilmir.

**Mənbələr və orijinal fayl adları:**

- [AYNA: velosiped zolaqları](https://opendata.az/en/@azerbaycan-yerustu-neqliyyat-agentliyi/velosiped-zolaqlarinin-cografi-koordinatlari)
  — `movcud-zolaqlar-velo.kmz`, yalnız **Mövcud zolaqlar** qatı.
- [AYNA: velosiped parkinqləri](https://opendata.az/@azerbaycan-yerustu-neqliyyat-agentliyi/velosiped-parkinq-menteqelerinin-cografi-koordinatlari)
  — `mpv-v-mps.-son.kmz`, yalnız **velosiped məntəqələri yeni** qatı.

**Metod:** KML koordinatları `(uzunluq, enlik, hündürlük)` sırasındadır.
Uzunluq və məsafə hesablamaları üçün ilk iki koordinat WGS 84-dən
**EPSG:32639 — WGS 84 / UTM zone 39N** sisteminə çevrilir. Nəticələr metrlə
hesablanır; yuvarlaqlaşdırma yalnız təqdimat zamanı edilir.
Parkinqin məsafəsi bütün xəttin ən yaxın nöqtəsinə qədər olan düzxətli,
ikiölçülü məsafədir. Bu göstərici yol üzrə məsafəni və ya təhlükəsiz fiziki
çıxışı təsdiqləmir.

Notebook-u hər iki KMZ faylı ilə eyni qovluqda açıb **Restart Kernel → Run All**
seçin. Təhvil zamanı notebook-un adını sənəddə tələb olunan
`Komanda_Rəhbər.ipynb` formatına uyğunlaşdırın; komanda və rəhbər adı təqdim edilməyib.

## 1. Mühit və faylların yoxlanması

Əsas hesablamalar üçün `pyproj` və `shapely`, interaktiv xəritələr üçün `folium`
istifadə edilir. Paketlər quraşdırılmayıbsa, ayrıca hüceyrədə bu əmri bir dəfə işlədin:

```python
%pip install pyproj==3.8.0 shapely==2.2.0 folium==0.20.0
```

Aşağıdakı kod faylların SHA-256 izlərini və istifadə olunan versiyaları çıxarır.
Beləliklə, sonradan məlumatlar dəyişərsə, fərqli nəticələrin səbəbi yoxlanıla bilər.

In [1]:
from pathlib import Path
from zipfile import ZipFile
from collections import Counter, defaultdict
from hashlib import sha256
from importlib.metadata import version
from html import escape
from math import fsum, hypot, isfinite
import platform
import xml.etree.ElementTree as ET

from pyproj import Transformer, Geod
from shapely.geometry import LineString, Point, mapping
from shapely.ops import unary_union
from shapely import transform as transform_geometry
from IPython.display import HTML, Markdown, display
import folium

DATA_DIR = Path.cwd()
LANES_FILE = DATA_DIR / 'movcud-zolaqlar-velo.kmz'
PARKING_FILE = DATA_DIR / 'mpv-v-mps.-son.kmz'
LANES_LAYER = 'Mövcud zolaqlar'
PARKING_LAYER = 'velosiped məntəqələri yeni'
DISTANCE_THRESHOLD_M = 50.0
# Sərhədlər daxildir: qərb, cənub, şərq, şimal.
BAKU_BOUNDS = (49.70, 40.30, 50.05, 40.46)

print('Python:', platform.python_version())
for package in ('pyproj', 'shapely', 'folium'):
    print(f'{package}: {version(package)}')
for path in (LANES_FILE, PARKING_FILE):
    if not path.is_file():
        raise FileNotFoundError(f'KMZ faylı tapılmadı: {path}. Notebook-u məlumat qovluğunda işlədin.')
    print(f'{path.name}: {path.stat().st_size:,} bytes; SHA-256: {sha256(path.read_bytes()).hexdigest()}')

Python: 3.14.7
pyproj: 3.8.0
shapely: 2.2.0
folium: 0.20.0
movcud-zolaqlar-velo.kmz: 11,816 bytes; SHA-256: dfbc6d1610a96db9c1bb466986175135c1e0a54bbf3213ac4cf81df60c71ac67
mpv-v-mps.-son.kmz: 87,908 bytes; SHA-256: 494065929eb8126a266699e7565d614218042b4c00b8da1902df9c8b746ee38e


In [2]:
NS = {'k': 'http://www.opengis.net/kml/2.2'}
projector = Transformer.from_crs('EPSG:4326', 'EPSG:32639', always_xy=True)
inverse_projector = Transformer.from_crs('EPSG:32639', 'EPSG:4326', always_xy=True)

def read_kml(path):
    # Arxivdən diskə heç nə çıxarılmır.
    with ZipFile(path) as archive:
        return ET.fromstring(archive.read('doc.kml'))

def get_layer(root, layer_name):
    folders = [
        folder for folder in root.findall('.//k:Folder', NS)
        if folder.findtext('k:name', default='', namespaces=NS).strip() == layer_name
    ]
    if len(folders) != 1:
        raise ValueError(f'{layer_name!r} qatı üçün {len(folders)} uyğunluq tapıldı; dəqiq bir qat gözlənilir.')
    return folders[0]

def read_coordinates(element):
    coordinates = []
    for token in (element.text or '').split():
        parts = token.split(',')
        if len(parts) < 2:
            raise ValueError(f'Natamam KML koordinatı: {token!r}')
        lon, lat = float(parts[0]), float(parts[1])
        if not (isfinite(lon) and isfinite(lat) and -180 <= lon <= 180 and -90 <= lat <= 90):
            raise ValueError(f'Etibarsız WGS 84 koordinatı: {token!r}')
        coordinates.append((lon, lat))
    return coordinates

def show_table(headers, rows):
    # Adlar HTML daxilində təhlükəsiz göstərilir.
    head = ''.join(f'<th>{escape(str(value))}</th>' for value in headers)
    body = ''.join(
        '<tr>' + ''.join(f'<td>{escape(str(value))}</td>' for value in row) + '</tr>'
        for row in rows
    )
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

lane_root = read_kml(LANES_FILE)
parking_root = read_kml(PARKING_FILE)
for path, root in ((LANES_FILE, lane_root), (PARKING_FILE, parking_root)):
    print(path.name)
    for folder in root.findall('.//k:Folder', NS):
        name = folder.findtext('k:name', default='', namespaces=NS)
        print(f'  {name}: {len(folder.findall("k:Placemark", NS))} birbaşa obyekt')

movcud-zolaqlar-velo.kmz
  VELO YOL 2026: 0 birbaşa obyekt
  Mövcud zolaqlar: 61 birbaşa obyekt
mpv-v-mps.-son.kmz
  Temporary Places: 0 birbaşa obyekt
  Temporary Places: 0 birbaşa obyekt
  Velosiped yolları 2025 yanvar: 45 birbaşa obyekt
  My Places: 649 birbaşa obyekt
  Sightseeing Tour: 10 birbaşa obyekt
  velosiped məntəqələri yeni: 185 birbaşa obyekt


## 2. Yalnız tapşırıqda göstərilən qatların oxunması

Parkinq arxivində köhnə velosiped yolları, `My Places` və turist obyektləri də var.
Bütün arxivin oxunması 185 parkinq əvəzinə başqa obyektləri də hesablamaya qata bilər.
Buna görə qat adları dəqiq seçilir, xətt və nöqtə koordinatları isə öz geometriya
elementlərindən oxunur. KML-dəki `LookAt` və `Camera` koordinatları istifadə edilmir.

Mövcud zolaqlarda koordinatsız **8 Noyabr pr.** qeydi ayrıca göstərilir və
uzunluq hesabına daxil edilmir. Digər 8 Noyabr xətləri hesablanır.
Küçə adları mənbədəki yazılışla saxlanılır; eyni adlı hissələr birləşdirilərək
uzunluqları toplanır. Müxtəlif yazılış variantları avtomatik eyniləşdirilmir.

In [3]:
lane_placemarks = get_layer(lane_root, LANES_LAYER).findall('k:Placemark', NS)
parking_placemarks = get_layer(parking_root, PARKING_LAYER).findall('k:Placemark', NS)
lanes, parking, excluded_lanes = [], [], []

for index, placemark in enumerate(lane_placemarks, start=1):
    name = placemark.findtext('k:name', default='', namespaces=NS).strip()
    line_elements = placemark.findall('.//k:LineString', NS)
    if len(line_elements) != 1:
        raise ValueError(f'{name}: bir LineString gözlənilir, {len(line_elements)} tapıldı.')
    coordinates_element = line_elements[0].find('k:coordinates', NS)
    if coordinates_element is None:
        excluded_lanes.append((index, name, 'Koordinat elementi yoxdur'))
        continue
    coordinates = read_coordinates(coordinates_element)
    if not coordinates:
        excluded_lanes.append((index, name, 'Koordinatlar boşdur'))
        continue
    if len(coordinates) < 2:
        raise ValueError(f'{name}: xətt üçün ən azı iki koordinat tələb olunur.')
    projected = [projector.transform(lon, lat) for lon, lat in coordinates]
    geometry = LineString(projected)
    if geometry.is_empty or not geometry.is_valid or geometry.length <= 0:
        raise ValueError(f'{name}: etibarsız xətt geometriyası.')
    lanes.append({'index': index, 'name': name, 'coordinates': coordinates,
                  'projected': projected, 'geometry': geometry, 'length_m': geometry.length})

west, south, east, north = BAKU_BOUNDS
for index, placemark in enumerate(parking_placemarks, start=1):
    name = placemark.findtext('k:name', default='', namespaces=NS).strip()
    point_elements = placemark.findall('.//k:Point', NS)
    if len(point_elements) != 1:
        raise ValueError(f'{name}: bir Point gözlənilir, {len(point_elements)} tapıldı.')
    coordinates_element = point_elements[0].find('k:coordinates', NS)
    if coordinates_element is None:
        raise ValueError(f'{name}: nöqtə koordinatları yoxdur.')
    coordinates = read_coordinates(coordinates_element)
    if len(coordinates) != 1:
        raise ValueError(f'{name}: dəqiq bir nöqtə koordinatı gözlənilir.')
    lon, lat = coordinates[0]
    geometry = Point(*projector.transform(lon, lat))
    parking.append({'index': index, 'name': name, 'lon': lon, 'lat': lat,
                    'geometry': geometry,
                    'inside_baku': west <= lon <= east and south <= lat <= north})

if len(parking) != 185:
    raise ValueError(f'Tapşırıqda 185 parkinq gözlənilir, faylda {len(parking)} tapıldı.')
if not lanes:
    raise ValueError('Hesablamaq üçün etibarlı velosiped zolağı yoxdur.')

show_table(['Göstərici', 'Say'], [
    ['Mövcud zolaqlar qatında bütün qeydlər', len(lane_placemarks)],
    ['Hesablanan xəttlər', len(lanes)],
    ['Boş olduğu üçün çıxarılan qeydlər', len(excluded_lanes)],
    ['Seçilmiş qatda parkinqlər', len(parking)],
    ['Təkrarlanan parkinq adları', sum(count - 1 for count in Counter(p['name'] for p in parking).values())],
    ['Təkrarlanan parkinq koordinatları', sum(count - 1 for count in Counter((p['lon'], p['lat']) for p in parking).values())],
])
show_table(['Qeyd №', 'Ad', 'Səbəb'], excluded_lanes)

Göstərici,Say
Mövcud zolaqlar qatında bütün qeydlər,61
Hesablanan xəttlər,60
Boş olduğu üçün çıxarılan qeydlər,1
Seçilmiş qatda parkinqlər,185
Təkrarlanan parkinq adları,0
Təkrarlanan parkinq koordinatları,0


Qeyd №,Ad,Səbəb
36,8 Noyabr pr.,Koordinatlar boşdur


## 3. Dəqiq geofəza hesablamaları və cavablar

**1.1:** Hər mövcud xəttin EPSG:32639 uzunluğu toplanır. Bu, verilən
zolaq geometriyalarının cəmidir; prospektin iki tərəfindəki ayrıca zolaqlar da
ayrıca hesablanır. Eyni küçənin bir neçə hissəsi üzrə uzunluqlar toplanır.

**1.2:** Bütün 185 parkinq üçün mövcud xətlərin birləşməsinə minimum məsafə
hesablanır və **məsafə ≤ 50 metr** şərti yoxlanır. Sərhəddəki qərar
yuvarlaqlaşdırılmamış məsafə ilə verilir. Koordinat çərçivəsindən kənardakı
nöqtələr bu sualın hesablamasından çıxarılmır.

**1.3:** Enlik **40.30–40.46**, uzunluq **49.70–50.05** çərçivəsindən
kənarda qalan nöqtələr seçilir. Sərhəd üzərindəki nöqtələr çərçivəyə daxildir.

In [4]:
network = unary_union([lane['geometry'] for lane in lanes])
total_length_m = fsum(lane['length_m'] for lane in lanes)
street_parts = defaultdict(list)
for lane in lanes:
    street_parts[lane['name']].append(lane['length_m'])
street_lengths_m = {name: fsum(parts) for name, parts in street_parts.items()}
street_ranking = sorted(street_lengths_m.items(), key=lambda item: (-item[1], item[0]))
longest_street, longest_street_m = street_ranking[0]

for point in parking:
    point['distance_m'] = point['geometry'].distance(network)
    point['within_50m'] = point['distance_m'] <= DISTANCE_THRESHOLD_M
    nearest_lane = min(lanes, key=lambda lane: point['geometry'].distance(lane['geometry']))
    point['nearest_street'] = nearest_lane['name']

nearby_parking = [point for point in parking if point['within_50m']]
outside_parking = [point for point in parking if not point['inside_baku']]
local_orphans = [point for point in parking if point['inside_baku'] and not point['within_50m']]

answer_1_1 = f'{total_length_m / 1000:.2f}, {longest_street}'
answer_1_2 = str(len(nearby_parking))
answer_1_3 = f'{len(outside_parking)}: ' + ', '.join(point['name'] for point in outside_parking)
print('Sual 1.1:', answer_1_1)
print('Sual 1.2:', answer_1_2)
print('Sual 1.3:', answer_1_3)
print()
print(f'Dəqiq ümumi uzunluq: {total_length_m:.9f} m')
print(f'{longest_street} üzrə cəm: {longest_street_m / 1000:.6f} km')
print(f'50 m daxilində parkinqlərin payı: {len(nearby_parking) / len(parking):.2%}')

Sual 1.1: 57.14, 8 Noyabr pr.
Sual 1.2: 87
Sual 1.3: 4: MPV-144-2, MPV-145-2, MPV-146-2, MPV-147-2

Dəqiq ümumi uzunluq: 57138.135770910 m
8 Noyabr pr. üzrə cəm: 13.856291 km
50 m daxilində parkinqlərin payı: 47.03%


In [5]:
show_table(['Küçə / prospekt — ilk 10', 'Zolaqların cəmi, km'], [
    [name, f'{length / 1000:.6f}'] for name, length in street_ranking[:10]
])
show_table(['Çərçivədən kənar parkinq', 'Enlik', 'Uzunluq', 'Ən yaxın zolağa məsafə, km'], [
    [point['name'], f'{point["lat"]:.8f}', f'{point["lon"]:.8f}', f'{point["distance_m"] / 1000:.3f}']
    for point in outside_parking
])
print('50 m sərhədinə ən yaxın 10 nöqtə:')
show_table(['Parkinq', 'Məsafə, m', '≤ 50 m', 'Ən yaxın küçə'], [
    [point['name'], f'{point["distance_m"]:.6f}', 'Bəli' if point['within_50m'] else 'Xeyr', point['nearest_street']]
    for point in sorted(parking, key=lambda point: abs(point['distance_m'] - DISTANCE_THRESHOLD_M))[:10]
])

Küçə / prospekt — ilk 10,"Zolaqların cəmi, km"
8 Noyabr pr.,13.856291
Neftçilər pr-Bulvar,2.262581
Xətai pr.,2.015171
28 may küç.,1.935430
Mətbuat pr.,1.907445
Hüseyn Cavid pr.,1.837750
Bulvar 2,1.628406
Rəşid Behbudov küç.,1.627306
Ağ şəhər bulvarı ərazisi,1.562978
Mirzəağa Əliyev küç.,1.501500


Çərçivədən kənar parkinq,Enlik,Uzunluq,"Ən yaxın zolağa məsafə, km"
MPV-144-2,40.59187336,49.98242376,24.264
MPV-145-2,40.58852420,49.97010673,23.507
MPV-146-2,40.57853941,49.93778173,21.536
MPV-147-2,40.58280500,49.93628131,21.954


50 m sərhədinə ən yaxın 10 nöqtə:


Parkinq,"Məsafə, m",≤ 50 m,Ən yaxın küçə
MPV-55-2,45.705166,Bəli,Vaqif pr.
MPV-99-1,54.967316,Xeyr,Mikayıl Müşviq küç.
MPV-126-2,44.156900,Bəli,Mətbuat pr.
MPV-84-2,42.596857,Bəli,Abdulla Şaiq küçəsi
MPV-21-2,42.549552,Bəli,Neftçilər pr-Bulvar
MPV-54-2,58.081692,Xeyr,Vaqif pr.
MPV-172-2,64.721163,Xeyr,Mirzəağa Əliyev küç.
MPV-51-2,34.744577,Bəli,İnşaatçılar pr.
MPV-61-2,33.355332,Bəli,Nəcəfqulu Rəfiyev küç.
MPV-69-2,29.082009,Bəli,Üzeyir H - Keykab S. - keçid


## 4. Hesablamaların yoxlanması

Uzunluqlar koordinat cütləri üzrə Evklid məsafələrinin cəmi ilə yenidən
hesablanır. Hər parkinqin minimum məsafəsi də bütün xətt seqmentlərinə ayrıca
proyeksiya ilə yoxlanır. Bu ikinci hesablama Shapely-nin məsafə funksiyasından
istifadə etmir və yalnız xətt təpələrinə qədər məsafə götürülmədiyini yoxlayır.

WGS 84 ellipsoidi üzrə uzunluq müqayisə üçün göstərilir; tapşırığın cavabı
məhz **EPSG:32639** nəticəsidir. Boş xəttin uzunluğu müəyyən edilə bilmədiyindən,
nəticə koordinatları verilmiş 60 xəttin cəmidir.

In [6]:
segment_lengths_m = fsum(
    hypot(b[0] - a[0], b[1] - a[1])
    for lane in lanes for a, b in zip(lane['projected'], lane['projected'][1:])
)
if abs(segment_lengths_m - total_length_m) > 1e-6:
    raise AssertionError('Seqment uzunluqları və geometriya uzunluqları uyğun gəlmir.')

def point_to_segment_distance(point, start, end):
    dx, dy = end[0] - start[0], end[1] - start[1]
    squared_length = dx * dx + dy * dy
    if squared_length == 0:
        return hypot(point[0] - start[0], point[1] - start[1])
    t = ((point[0] - start[0]) * dx + (point[1] - start[1]) * dy) / squared_length
    t = min(1.0, max(0.0, t))
    return hypot(point[0] - (start[0] + t * dx), point[1] - (start[1] + t * dy))

segments = [
    (start, end) for lane in lanes
    for start, end in zip(lane['projected'], lane['projected'][1:])
]
maximum_distance_difference_m = 0.0
for point in parking:
    coordinates = (point['geometry'].x, point['geometry'].y)
    independent_distance = min(point_to_segment_distance(coordinates, start, end) for start, end in segments)
    maximum_distance_difference_m = max(maximum_distance_difference_m, abs(independent_distance - point['distance_m']))
    if abs(independent_distance - point['distance_m']) > 1e-6:
        raise AssertionError(f'{point["name"]}: məsafə yoxlaması uyğun gəlmir.')
    if (independent_distance <= DISTANCE_THRESHOLD_M) != point['within_50m']:
        raise AssertionError(f'{point["name"]}: 50 m təsnifatı uyğun gəlmir.')

duplicate_lanes = [
    (lanes[i]['name'], lanes[j]['name'])
    for i in range(len(lanes)) for j in range(i)
    if lanes[i]['geometry'].equals(lanes[j]['geometry'])
]
geod = Geod(ellps='WGS84')
geodesic_length_m = fsum(
    geod.line_length(*zip(*lane['coordinates'])) for lane in lanes
)
print(f'Uzunluq yoxlamasının fərqi: {abs(segment_lengths_m - total_length_m):.12f} m')
print(f'{len(parking)} parkinq üçün maksimum məsafə fərqi: {maximum_distance_difference_m:.12f} m')
print(f'Eyni geometriyalı təkrar xəttlər: {len(duplicate_lanes)}')
print(f'Xətt uzunluqları cəmi ilə həndəsi birləşmənin uzunluq fərqi: {abs(total_length_m - network.length):.9f} m')
print(f'Müqayisə üçün geodezik uzunluq: {geodesic_length_m / 1000:.6f} km')
print(f'Tələb olunan EPSG:32639 uzunluğu: {total_length_m / 1000:.6f} km')
print('Yoxlamalar uğurla tamamlandı.')

Uzunluq yoxlamasının fərqi: 0.000000000007 m
185 parkinq üçün maksimum məsafə fərqi: 0.000000000658 m
Eyni geometriyalı təkrar xəttlər: 0
Xətt uzunluqları cəmi ilə həndəsi birləşmənin uzunluq fərqi: 0.000000000 m
Müqayisə üçün geodezik uzunluq: 57.154414 km
Tələb olunan EPSG:32639 uzunluğu: 57.138136 km
Yoxlamalar uğurla tamamlandı.


## 5. “Tənha” parkinqlər və növbəti mərhələnin prioriteti

Bu təhlildə **“tənha”** parkinq mövcud zolaqdan **50 metrdən uzaq** nöqtədir.
Şəhər çərçivəsindəki nöqtələr şəbəkənin genişləndirilməsi üçün ayrıca
qiymətləndirilir. Çərçivədən kənardakı dörd nöqtə **21–25 km** məsafədədir;
onların həqiqi yerləşməsi yoxlanmalıdır. Çərçivədən kənar olmaq özü-özlüyündə
koordinat səhvini sübut etmir.

Ən yaxın mövcud küçə üzrə bölgü ilk bağlantıların hansı istiqamətlərdən
başlanmasını göstərir. Şimal üçün enlik **40.39** yalnız təsviri bölgü həddidir;
klaster analizi və ya inzibati sərhəd deyil.

In [7]:
orphan_total = sum(not point['within_50m'] for point in parking)
northern_local_orphans = sum(point['lat'] > 40.39 for point in local_orphans)
nearest_street_counts = Counter(point['nearest_street'] for point in local_orphans)
short_connection_points = [point for point in local_orphans if point['distance_m'] <= 250]
bands = [(0, 50, '0–50'), (50, 250, '>50–250'), (250, 500, '>250–500'),
         (500, 1000, '>500–1000'), (1000, 2000, '>1000–2000'), (2000, float('inf'), '>2000')]

show_table(['Göstərici', 'Say'], [
    ['185 parkinqdən 50 m daxilində olanlar', len(nearby_parking)],
    ['Bütün tənha parkinqlər', orphan_total],
    ['Şəhər çərçivəsində tənha parkinqlər', len(local_orphans)],
    ['Şəhər çərçivəsindən kənar parkinqlər', len(outside_parking)],
    ['40.39 enliyindən şimaldakı yerli tənha parkinqlər', northern_local_orphans],
    ['Şəhər üzrə 50–250 m məsafədəki qısa bağlantı namizədləri', len(short_connection_points)],
])
show_table(['Ən yaxın mövcud küçə / prospekt — ilk 8', 'Yerli tənha parkinqlər'],
           nearest_street_counts.most_common(8))
show_table(['Məsafə intervalı, m', 'Bütün parkinqlər'], [
    [label, sum((point['distance_m'] >= low if low == 0 else point['distance_m'] > low)
                and point['distance_m'] <= high for point in parking)]
    for low, high, label in bands
])

Göstərici,Say
185 parkinqdən 50 m daxilində olanlar,87
Bütün tənha parkinqlər,98
Şəhər çərçivəsində tənha parkinqlər,94
Şəhər çərçivəsindən kənar parkinqlər,4
40.39 enliyindən şimaldakı yerli tənha parkinqlər,53
Şəhər üzrə 50–250 m məsafədəki qısa bağlantı namizədləri,23


Ən yaxın mövcud küçə / prospekt — ilk 8,Yerli tənha parkinqlər
Vaqif pr.,27
Mətbuat pr.,14
Xətai pr.,13
8 Noyabr pr.,13
Şəfayət Mehdiyev küç.,7
Xocalı pr.,3
Mirzəağa Əliyev küç.,3
Bülbül pr.,2


"Məsafə intervalı, m",Bütün parkinqlər
0–50,87
>50–250,23
>250–500,14
>500–1000,15
>1000–2000,23
>2000,23


### AYNA rəhbərliyinə təklif — maksimum 99 söz

Təklif koordinatların paylanmasından çıxarılan **ilkin strateji qiymətləndirmədir**.
Fayllarda tam küçə şəbəkəsi, yol eni, maillik, keçidlər, tikinti xərcləri və
icra müddətləri yoxdur. Buna görə bu məlumatlardan ən sürətli və ya ən ucuz
dəqiq marşrutu sübut etmək mümkün deyil. 27 nöqtənin hamısının tək Azadlıq
zolağı ilə 50 m daxilinə düşəcəyi iddia edilmir; yan bağlantılar tələb oluna bilər.

In [8]:
recommendation = (
    f'Vaqif prospekti ilə bağlantıdan başlayaraq Azadlıq prospekti boyunca şimal istiqamətində '
    f'velosiped zolağı təklif edirəm. Şəhər çərçivəsindəki {len(local_orphans)} “tənha” parkinqdən '
    f'{nearest_street_counts["Vaqif pr."]}-sinin ən yaxın mövcud zolağı Vaqif prospektindədir; '
    f'bu, həmin ərazidə şəbəkənin genişləndirilməsinə əsas verir. Mərhələli tikinti və qısa '
    f'yan bağlantılar parkinqlərin qoşulmasını sürətləndirə bilər. Çərçivədən kənardakı '
    f'dörd məntəqənin yerləşməsi ayrıca yoxlanılmalıdır. Bu, ilkin prioritetdir; '
    f'icra sürətini müəyyənləşdirmək üçün yol eni, keçidlər və tikinti xərcləri yerində qiymətləndirilməlidir.'
)
word_count = len(recommendation.split())
if word_count > 99:
    raise AssertionError(f'Təklif {word_count} sözdür; limit 99-dur.')
display(Markdown(recommendation))
print(f'Söz sayı: {word_count} / 99')

Vaqif prospekti ilə bağlantıdan başlayaraq Azadlıq prospekti boyunca şimal istiqamətində velosiped zolağı təklif edirəm. Şəhər çərçivəsindəki 94 “tənha” parkinqdən 27-sinin ən yaxın mövcud zolağı Vaqif prospektindədir; bu, həmin ərazidə şəbəkənin genişləndirilməsinə əsas verir. Mərhələli tikinti və qısa yan bağlantılar parkinqlərin qoşulmasını sürətləndirə bilər. Çərçivədən kənardakı dörd məntəqənin yerləşməsi ayrıca yoxlanılmalıdır. Bu, ilkin prioritetdir; icra sürətini müəyyənləşdirmək üçün yol eni, keçidlər və tikinti xərcləri yerində qiymətləndirilməlidir.

Söz sayı: 65 / 99


## 6. İnteraktiv xəritələr

Birinci xəritə şəhər daxilində zolaqları, 50 m əhatə sahəsini və parkinqləri göstərir.
**Yaşıl:** zolağa ≤ 50 m. **Narıncı:** şəhər daxilində, zolaqdan > 50 m.
Nöqtəyə klik edərək adı, dəqiq məsafəni və ən yaxın küçəni görmək olar.

İkinci xəritə çərçivədən kənardakı **qırmızı** nöqtələri və tapşırığın
koordinat çərçivəsini göstərir. Beləliklə, uzaq nöqtələr şəhər xəritəsinin
miqyasını kiçiltmir. Xəritə fonu və brauzer kitabxanaları internet tələb edir;
rəqəmsal hesablamalar internetdən asılı deyil. 50 m sahəsi yalnız xəritədə
göstərmək üçün 2 m tolerantlıqla sadələşdirilib; məsafələr tam geometriyalardan hesablanıb.

In [9]:
city_map = folium.Map(location=[40.39, 49.85], tiles='OpenStreetMap', zoom_start=12)
lane_group = folium.FeatureGroup(name='Mövcud velosiped zolaqları', show=True).add_to(city_map)
for lane in lanes:
    folium.PolyLine(
        [(lat, lon) for lon, lat in lane['coordinates']], color='#2563eb', weight=3,
        tooltip=f'{escape(lane["name"])} — {lane["length_m"] / 1000:.3f} km'
    ).add_to(lane_group)

coverage_group = folium.FeatureGroup(name='50 m məsafə sahəsi', show=False).add_to(city_map)
coverage_wgs84 = transform_geometry(
    network.buffer(DISTANCE_THRESHOLD_M).simplify(2.0, preserve_topology=True),
    inverse_projector.transform, interleaved=False
)
folium.GeoJson(
    mapping(coverage_wgs84),
    style_function=lambda feature: {'fillColor': '#16a34a', 'color': '#16a34a',
                                    'weight': 1, 'fillOpacity': 0.15}
).add_to(coverage_group)

connected_group = folium.FeatureGroup(name=f'≤ 50 m: {len(nearby_parking)} parkinq', show=True).add_to(city_map)
orphan_group = folium.FeatureGroup(name=f'> 50 m: {len(local_orphans)} yerli parkinq', show=True).add_to(city_map)
for point in parking:
    if not point['inside_baku']:
        continue
    color = '#16a34a' if point['within_50m'] else '#ea580c'
    group = connected_group if point['within_50m'] else orphan_group
    popup = (f'<b>{escape(point["name"])}</b><br>'
             f'Məsafə: {point["distance_m"]:.2f} m<br>'
             f'Ən yaxın küçə: {escape(point["nearest_street"])}')
    folium.CircleMarker([point['lat'], point['lon']], radius=4, color=color,
                        fill=True, fill_color=color, fill_opacity=0.9,
                        tooltip=point['name'], popup=folium.Popup(popup, max_width=300)).add_to(group)
folium.LayerControl(collapsed=False).add_to(city_map)
city_coordinates = [(lat, lon) for lane in lanes for lon, lat in lane['coordinates']]
city_coordinates += [(point['lat'], point['lon']) for point in parking if point['inside_baku']]
city_map.fit_bounds(city_coordinates)
display(city_map)

In [10]:
overview_map = folium.Map(location=[40.45, 49.86], tiles='OpenStreetMap', zoom_start=10)
folium.Rectangle(
    bounds=[[south, west], [north, east]], color='#2563eb', weight=2, fill=False,
    tooltip='Tapşırığın Bakı çərçivəsi: enlik 40.30–40.46, uzunluq 49.70–50.05'
).add_to(overview_map)
for lane in lanes:
    folium.PolyLine([(lat, lon) for lon, lat in lane['coordinates']],
                    color='#2563eb', weight=2, opacity=0.7).add_to(overview_map)
for point in outside_parking:
    folium.CircleMarker(
        [point['lat'], point['lon']], radius=7, color='#dc2626', fill=True,
        fill_opacity=0.95, tooltip=point['name'],
        popup=f'{escape(point["name"])}<br>Enlik: {point["lat"]:.8f}<br>Uzunluq: {point["lon"]:.8f}'
    ).add_to(overview_map)
overview_map.fit_bounds([[south, west], [north, east]] +
                       [[point['lat'], point['lon']] for point in outside_parking])
display(overview_map)

## 7. Parkinqlər üzrə tam nəticə cədvəli

Aşağıdakı cədvəl hər parkinqin təsnifatını yoxlamağa imkan verir. Cədvəldə
göstərilən məsafə üç rəqəmə yuvarlaqlaşdırılıb; ≤ 50 m şərti tam dəqiqliklə hesablanıb.

In [11]:
show_table(['Parkinq', 'Enlik', 'Uzunluq', 'Məsafə, m', '≤ 50 m', 'Bakı çərçivəsində', 'Ən yaxın küçə'], [
    [point['name'], f'{point["lat"]:.8f}', f'{point["lon"]:.8f}', f'{point["distance_m"]:.3f}',
     'Bəli' if point['within_50m'] else 'Xeyr', 'Bəli' if point['inside_baku'] else 'Xeyr', point['nearest_street']]
    for point in parking
])

Parkinq,Enlik,Uzunluq,"Məsafə, m",≤ 50 m,Bakı çərçivəsində,Ən yaxın küçə
MPV-1-2,40.36577613,49.83122243,5.265,Bəli,Bəli,Filarmoniya bağı.
MPV-2-2,40.36897587,49.83476438,10.998,Bəli,Bəli,İstiqlaliyyət küç.
MPV-3-2,40.36945891,49.83468747,3.767,Bəli,Bəli,Azərbaycan pr.
MPV-4-2,40.37107147,49.83397958,0.285,Bəli,Bəli,Azərbaycan pr.
MPV-5-2,40.37421575,49.83261468,3.542,Bəli,Bəli,Azərbaycan pr.
MPV-6-2,40.37513910,49.83402316,12.845,Bəli,Bəli,Mirzəağa Əliyev küç.
MPV-7-2,40.37540580,49.84320296,0.476,Bəli,Bəli,Bülbül pr.
MPV-8-3,40.37262186,49.84456802,11.904,Bəli,Bəli,Bülbül pr.
MPV-9-2,40.37904229,49.84730574,10.229,Bəli,Bəli,Azadlıq pr.
MPV-10-2,40.38374444,49.84244438,11.612,Bəli,Bəli,Rəşid Behbudov küç.
